In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
import time
from langgraph.checkpoint.memory import InMemorySaver # This stores state in RAM

In [2]:
class State(TypedDict):
    input:str
    step_1:str
    step_2:str
    step_3:str
    

In [3]:
def step_1(state:State):
    print("Step_1")
    return {"step_1":"done"}

def step_2(state:State):
    print('Sleeping...')
    time.sleep(10)
    print("Step_2:Done")
    
    return {"step_2":"done"}

def step_3(state:State):
    print("Step_3")
    
    return {"step_3":"done"}

In [4]:
graph = StateGraph(State)

graph.add_node("step_1",step_1)
graph.add_node("step_2",step_2)
graph.add_node("step_3",step_3)

graph.add_edge(START, 'step_1')
graph.add_edge("step_1", 'step_2')
graph.add_edge("step_2", 'step_3')
graph.add_edge("step_3", END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [5]:
config = {'configurable':{"thread_id":"1"}}

initial_state = {"input":"Start"}
workflow.invoke(initial_state, config=config)

Step_1
Sleeping...


KeyboardInterrupt: 

In [6]:
workflow.get_state(config=config)

StateSnapshot(values={'input': 'Start', 'step_1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca88-2bae-6a7e-8001-aeafb1fee053'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-03-10T17:42:29.856706+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca88-2ba5-6eae-8000-c2fdb0e46cd1'}}, tasks=(PregelTask(id='494e35e0-80e8-975a-1e7f-19296547df1f', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

### Resuming from last step

In [7]:
workflow.invoke(None, config=config) # When you pass None as initial state in checkpoint workflow, It started execution from last/prev state where node execution failed.

Sleeping...
Step_2:Done
Step_3


{'input': 'Start', 'step_1': 'done', 'step_2': 'done', 'step_3': 'done'}

In [8]:
workflow.get_state(config=config)


StateSnapshot(values={'input': 'Start', 'step_1': 'done', 'step_2': 'done', 'step_3': 'done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca89-364f-6cfb-8003-7f16cdeb7ade'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-03-10T17:42:57.814826+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca89-3649-6ccd-8002-19910ab0dae7'}}, tasks=(), interrupts=())

### Time Travel - Start workflow from a specific point in time

In [10]:
list(workflow.get_state_history(config=config))


[StateSnapshot(values={'input': 'Start', 'step_1': 'done', 'step_2': 'done', 'step_3': 'done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca89-364f-6cfb-8003-7f16cdeb7ade'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-03-10T17:42:57.814826+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca89-3649-6ccd-8002-19910ab0dae7'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'Start', 'step_1': 'done', 'step_2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca89-3649-6ccd-8002-19910ab0dae7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-10T17:42:57.812355+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca88-2bae-6a7e-8001-aeafb1fee053'}}, tasks=(PregelTask(id='790f1931-ea5e-6536-3b42-3429f895bc23', na

In [11]:
workflow.get_state(config={"configurable":{"thread_id":"1", "checkpoint_id":"1f11ca89-3649-6ccd-8002-19910ab0dae7"}}, )

StateSnapshot(values={'input': 'Start', 'step_1': 'done', 'step_2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f11ca89-3649-6ccd-8002-19910ab0dae7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-10T17:42:57.812355+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca88-2bae-6a7e-8001-aeafb1fee053'}}, tasks=(PregelTask(id='790f1931-ea5e-6536-3b42-3429f895bc23', name='step_3', path=('__pregel_pull', 'step_3'), error=None, interrupts=(), state=None, result={'step_3': 'done'}),), interrupts=())

In [ ]:
# You pass checkpoint_id from which step you want to start and pass None initial State
workflow.invoke(None, config={"configurable":{"thread_id":"1", "checkpoint_id":"1f11ca89-3649-6ccd-8002-19910ab0dae7"}})

Step_3


{'input': 'Start', 'step_1': 'done', 'step_2': 'done', 'step_3': 'done'}

In [13]:
list(workflow.get_state_history(config=config))


[StateSnapshot(values={'input': 'Start', 'step_1': 'done', 'step_2': 'done', 'step_3': 'done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca91-52df-6220-8003-52bd2746d12e'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-03-10T17:46:35.557899+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca89-3649-6ccd-8002-19910ab0dae7'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'Start', 'step_1': 'done', 'step_2': 'done', 'step_3': 'done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca89-364f-6cfb-8003-7f16cdeb7ade'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-03-10T17:42:57.814826+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca89-3649-6ccd-8002-19910ab0dae7'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'inpu

### Update State

In [16]:
workflow.update_state({"configurable":{"thread_id":"1", "checkpoint_id":"1f11ca88-2ba5-6eae-8000-c2fdb0e46cd1", "checkpoint_ns":""}}, {"input":"UpdatedInput"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f11ca9e-449a-6cc8-8001-c28ce6e49a24'}}

In [17]:
list(workflow.get_state_history(config=config))


[StateSnapshot(values={'input': 'UpdatedInput'}, next=('step_1',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca9e-449a-6cc8-8001-c28ce6e49a24'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-03-10T17:52:23.028001+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca88-2ba5-6eae-8000-c2fdb0e46cd1'}}, tasks=(PregelTask(id='ec8ee5a8-4c0e-9dbf-d442-8625af5835ad', name='step_1', path=('__pregel_pull', 'step_1'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'Start', 'step_1': 'done', 'step_2': 'done', 'step_3': 'done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11ca91-52df-6220-8003-52bd2746d12e'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-03-10T17:46:35.557899+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns'